# Multi-Step Prosumer Energy Forecasting

**Conference Paper + Journal Paper — unified pipeline**

| Section | Content |
|---------|---------|
| 1 | Environment setup |
| 2 | Configuration (all paths & params here) |
| 3 | Data loading & merging (Swedish + CityLearn) |
| 4 | Preprocessing, scaling & windowing |
| 5 | Multi-Step Forecasting: Direct · Recursive · Hybrid + **CityLearn** |
| 5b | Conference Paper Baselines: GP · LightGBM · CatBoost · LinearQR (Table 3) |
| 5c | §5.1 Training-Set Size Sweep (Figure 3) |
| 5d | §5.2 Aggregation Scenarios — Individual / Grid / LOO (Figure 4) |
| 5e | §5.3 Auto-Correlation Analysis (Figure 5) |
| 5f | §5.4 Feature-Target Correlation (Figure 6) |
| 6 | Evaluation Metrics (sMAPE, MQL_Σ, PICP, MPIR, CFE) |
| 7 | Visualisation |

**Dataset:** 7 Uppsala + 5 Halmstad prosumers + CityLearn 2022  
**Features:** energy consumption/production + weather  
**Horizon:** 48 h · **Input window:** 24 h · **Coverage target:** 90 %


## 1. Environment Setup

In [ ]:
# Install dependencies from requirements.txt (run once per environment).
# For an exact reproducible freeze, run afterwards:
#   pip freeze > requirements-lock.txt
!pip install -r requirements.txt --quiet


In [ ]:
import sys, os

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('Analysis.ipynb')) \
               if os.path.isabs('Analysis.ipynb') else os.getcwd()
sys.path.insert(0, os.path.join(NOTEBOOK_DIR, 'Helper_Functions'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import (
    merge_and_save_combined,
    build_prosumer_datasets,
    preprocess_scale_and_create_multistep,
)
from preprocessing_citylearn import (
    build_citylearn_datasets,
    preprocess_citylearn_multistep,
)
from models import (
    direct_multistep_forecast,
    recursive_multistep_forecast,
    hybrid_multistep_forecast,
)
from models_conference import (
    create_conference_splits,
    gp_forecast,
    lgbm_singlestep_forecast,
    catboost_quantile_forecast,
    catboost_multiquantile_forecast,
    linear_quantile_forecast,
    run_all_conference_models,
)
from evaluation import (
    save_forecast_results,
    calculate_smape,
    calculate_mql_sum,
    calculate_picp,
    calculate_mpir,
    compute_all_metrics,
    build_summary_table,
    generate_all_error_plots,
    plot_single_forecast,
)
from analysis import (
    training_size_sweep,
    plot_training_size_sweep,
    aggregation_scenarios,
    plot_aggregation_scenarios,
    autocorrelation_matrix,
    plot_autocorrelation_heatmap,
    feature_target_correlation,
    plot_correlation_heatmap,
    persistence_forecast,
)

print('Imports OK')


## 2. Configuration

All paths and hyper-parameters live here. Change once, applies everywhere.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
BASE_DIR    = NOTEBOOK_DIR
DATASET_DIR = os.path.join(BASE_DIR, 'Dataset')
RESULTS_DIR = os.path.join(BASE_DIR, 'Results')
METRICS_DIR = os.path.join(BASE_DIR, 'Results', 'Metrics')
IMAGES_DIR  = os.path.join(BASE_DIR, 'Images')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR,  exist_ok=True)

# ── Forecasting settings ──────────────────────────────────────────────────
INPUT_STEPS    = 24     # look-back window (hours)
FORECAST_STEPS = 48     # forecast horizon (hours)
COVERAGE       = 0.90   # target prediction interval coverage
TRAIN_RATIO    = 0.80   # train / test split

print(f'Dataset : {DATASET_DIR}')
print(f'Results : {RESULTS_DIR}')
print(f'Horizon : {FORECAST_STEPS} h  |  Window: {INPUT_STEPS} h  |  Coverage: {COVERAGE}')

## 3. Data Loading & Merging

> **Skip** if combined CSVs already exist in `Dataset/combined_datasets/`.

In [ ]:
# 3a. Merge raw per-prosumer CSVs with weather data
# Writes Dataset/combined_datasets/Uppsala_combined.csv  +  Halmstad_combined.csv
merge_and_save_combined(DATASET_DIR)

In [ ]:
# 3b. Build per-prosumer Consumption / Production CSVs
# Applies month-specific daylight-hours filter for production datasets
build_prosumer_datasets(DATASET_DIR)

### 3c. CityLearn 2022 Dataset

> **Place files** in `Dataset/ELAD_Data/` — Building_1.csv … Building_17.csv + weather.csv  
> (Publicly available at the CityLearn 2022 Challenge GitHub repository.)

Run `build_citylearn_datasets()` once to write the aggregated CSVs, then
`preprocess_citylearn_multistep()` to create windowed arrays compatible with Section 5.


In [ ]:
ELAD_DIR = os.path.join(DATASET_DIR, 'ELAD_Data')

if os.path.isdir(ELAD_DIR) and any(f.startswith('Building_') for f in os.listdir(ELAD_DIR)):
    # Step 1: write aggregated CSVs (run once)
    build_citylearn_datasets(
        elad_dir=ELAD_DIR,
        output_dir=os.path.join(DATASET_DIR, 'combined_datasets'),
    )
    # Step 2: create windowed multi-step arrays.
    # Each entry is a 5-tuple: (X_train, y_train, X_test, y_test, y_last_obs)
    citylearn_multistep = preprocess_citylearn_multistep(
        elad_dir=ELAD_DIR,
        input_steps=INPUT_STEPS,
        forecast_steps=FORECAST_STEPS,
        train_ratio=TRAIN_RATIO,
    )
    for key, lst in citylearn_multistep.items():
        Xtr, ytr, Xte, yte, yte_last = lst[0]
        print(f'{key}  X_train={Xtr.shape}  y_last_obs={yte_last.shape}')
else:
    citylearn_multistep = {}
    print(f'[skip] CityLearn ELAD_Data not found at {ELAD_DIR}')
    print('       Place Building_*.csv + weather.csv there and re-run.')


## 4. Preprocessing, Scaling & Multi-Step Windowing

In [ ]:
# Load all prosumer CSVs, add time features, scale, and create sliding windows.
# Each entry in data_list is a 5-tuple:
#   (X_train, y_train, X_test, y_test, y_test_last_obs)
# where y_test_last_obs[i] is the last observed target value before test window i.
multistep_data = preprocess_scale_and_create_multistep(
    DATASET_DIR,
    input_steps=INPUT_STEPS,
    forecast_steps=FORECAST_STEPS,
    train_ratio=TRAIN_RATIO,
)

for category, data_list in multistep_data.items():
    if data_list:
        Xtr, ytr, Xte, yte, yte_last = data_list[0]
        print(f'{category:30s}  X_train={Xtr.shape}  y_train={ytr.shape}')


## 5. Multi-Step Forecasting

Three strategies are implemented in `Helper_Functions/models.py`:

| Strategy | Description |
|----------|-------------|
| **Direct** | Independent LightGBM model per horizon step |
| **Recursive** | Single model; predictions fed back as features |
| **Hybrid** | Direct training, recursive test-feature augmentation |

> **Runtime:** each strategy runs grid-search (3-fold, 48 steps, 12 prosumers).  
> Expect ~30–60 min on CPU. Comment out strategies you don't need.

In [ ]:
def run_strategy(name, fn, include_citylearn=True):
    """Run a multi-step forecasting strategy over all datasets.

    Iterates over Swedish prosumers (multistep_data) and, if available,
    CityLearn (citylearn_multistep).  Each data_list entry is a 5-tuple:
    (X_train, y_train, X_test, y_test, y_last_obs) — the 5th element is
    used only by the persistence baseline, not by the strategy functions.
    """
    print(f'\n{"="*60}\n  {name.upper()}\n{"="*60}')

    # ── Swedish prosumers ────────────────────────────────────────────────
    for category, data_list in multistep_data.items():
        for idx, (Xtr, ytr, Xte, yte, _) in enumerate(data_list):
            print(f'  {category} [{idx}]')
            preds, lowers, uppers, y_actual = fn(
                Xtr, ytr, Xte, yte,
                input_steps=INPUT_STEPS,
                forecast_steps=FORECAST_STEPS,
                confidence_interval=COVERAGE,
            )
            out = os.path.join(RESULTS_DIR, f'{name}_{category}_{idx}.csv')
            save_forecast_results(preds, lowers, uppers, y_actual, out)

    # ── CityLearn ────────────────────────────────────────────────────────
    if include_citylearn and citylearn_multistep:
        for category, data_list in citylearn_multistep.items():
            for idx, (Xtr, ytr, Xte, yte, _) in enumerate(data_list):
                print(f'  {category} [{idx}]  (CityLearn)')
                preds, lowers, uppers, y_actual = fn(
                    Xtr, ytr, Xte, yte,
                    input_steps=INPUT_STEPS,
                    forecast_steps=FORECAST_STEPS,
                    confidence_interval=COVERAGE,
                )
                out = os.path.join(RESULTS_DIR, f'{name}_{category}_{idx}.csv')
                save_forecast_results(preds, lowers, uppers, y_actual, out)


In [ ]:
run_strategy('direct_forecast',    direct_multistep_forecast)

In [ ]:
run_strategy('recursive_forecast', recursive_multistep_forecast)

In [ ]:
run_strategy('hybrid_forecast',    hybrid_multistep_forecast)

### 5g. Persistence Baseline (§5.6 Table 4)

Naive benchmark: ŷ_{t+h} = y_t (last observed value repeated for all 48 steps).
Computed for every prosumer and CityLearn dataset.  Results are compared
against Direct / Recursive / Hybrid in Table 4 of the Journal Paper.


In [ ]:
persistence_rows = []

all_datasets = dict(multistep_data)
if citylearn_multistep:
    all_datasets.update(citylearn_multistep)

for category, data_list in all_datasets.items():
    for idx, (_, _, Xte, yte, yte_last) in enumerate(data_list):
        preds, lowers, uppers, y_test = persistence_forecast(
            yte, yte_last,
            forecast_steps=FORECAST_STEPS,
            confidence_fraction=COVERAGE,
        )
        out = os.path.join(RESULTS_DIR, f'persistence_{category}_{idx}.csv')
        save_forecast_results(preds, lowers, uppers, y_test, out)

        mae_mean    = np.mean(np.abs(y_test - preds))
        smape_avg   = np.mean([calculate_smape(preds[:, s], y_test[:, s])
                               for s in range(FORECAST_STEPS)])
        mpir_avg    = calculate_mpir(lowers.mean(0), uppers.mean(0))
        picp_val, _ = calculate_picp(lowers, uppers, y_test)
        cfe_val     = abs(COVERAGE - picp_val)
        persistence_rows.append({
            'category': category, 'idx': idx,
            'MAE': round(mae_mean, 4), 'sMAPE': round(smape_avg, 2),
            'MPIR': round(mpir_avg, 4), 'PICP': round(picp_val, 4),
            'CFE': round(cfe_val, 4),
        })

persistence_summary = pd.DataFrame(persistence_rows)
persistence_summary.to_csv(
    os.path.join(RESULTS_DIR, 'persistence_baseline_summary.csv'), index=False
)
display(persistence_summary)


## 5b. Conference Paper Baseline Models (Single-Step)

These models were used in the **Conference Paper** to produce the single-step
probabilistic forecasts for each prosumer independently (Scenario 1: per-prosumer).

| Model | Description |
|-------|-------------|
| **GP** | Gaussian Process (RBF + WhiteKernel + DotProduct) — may be slow, toggle `RUN_GP` |
| **LightGBM** | 3 quantile LGBMRegressors (lower / median / upper) |
| **CatBoost** | 3 separate CatBoost Quantile models |
| **CatBoost_Multi** | 1 CatBoost MultiQuantile model (deciles 0.1–0.9) |
| **LinearQR** | sklearn QuantileRegressor baseline |

> Results are saved to `Results/Conference_<model>_<category>_<prosumer>.csv`


In [ ]:
# ── Conference Paper settings ─────────────────────────────────────────────
LOWER_Q = 0.10
UPPER_Q = 0.90
RUN_GP  = False   # Set True to include Gaussian Process (slow on large datasets)

CONF_RESULTS_DIR = os.path.join(RESULTS_DIR, 'Conference')
os.makedirs(CONF_RESULTS_DIR, exist_ok=True)
print(f'Conference results → {CONF_RESULTS_DIR}')


In [ ]:
import glob

# Load per-prosumer combined CSVs (produced by Section 3 / preprocessing.py)
combined_dir = os.path.join(DATASET_DIR, 'combined_datasets')
prosumer_files = sorted(glob.glob(os.path.join(combined_dir, '*.csv')))
print(f'Found {len(prosumer_files)} prosumer CSV(s)')

# Build a simple list of DataFrames (one per prosumer) for the conference models
# Each DataFrame has Date as index, columns: energy + weather features
conf_dfs = []
for fpath in prosumer_files:
    df = pd.read_csv(fpath, parse_dates=['Date'], index_col='Date')
    # Drop columns that are not features (IDs etc.)
    drop_cols = [c for c in df.columns if c.lower() in ('unnamed: 0', 'id')]
    df = df.drop(columns=drop_cols, errors='ignore')
    conf_dfs.append((os.path.basename(fpath).replace('.csv', ''), df))

print('Prosumers loaded:', [name for name, _ in conf_dfs])


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

def run_conference_scenario(target_col, label, run_gp=RUN_GP):
    """Run all Conference Paper models for one target column across all prosumers.

    Saves one CSV per model × prosumer to Results/Conference/.
    Returns a summary DataFrame with all Table 3 metrics:
      MAE, R², sMAPE (%), MQL_Σ, MPIR, CFE.
    """
    all_rows = []

    for prosumer_name, df in conf_dfs:
        if target_col not in df.columns:
            continue

        X_train, Y_train, X_test, Y_test, scaler_y, _ = create_conference_splits(
            df, target_col
        )

        results = run_all_conference_models(
            X_train, Y_train, X_test, Y_test, scaler_y,
            lower_quantile=LOWER_Q, upper_quantile=UPPER_Q,
            run_gp=run_gp,
        )

        for model_name, res in results.items():
            y_true  = np.array(res['y_true'])
            y_pred  = np.array(res['y_pred'])
            y_lower = np.array(res['y_lower'])
            y_upper = np.array(res['y_upper'])

            # Save raw predictions
            out_path = os.path.join(
                CONF_RESULTS_DIR,
                f'{model_name}_{label}_{prosumer_name}.csv'
            )
            pd.DataFrame({
                'y_true': y_true, 'y_pred': y_pred,
                'y_lower': y_lower, 'y_upper': y_upper,
            }).to_csv(out_path, index=False)

            # ── Table 3 metrics ────────────────────────────────────────────
            mae   = mean_absolute_error(y_true, y_pred)
            r2    = r2_score(y_true, y_pred)
            smape = calculate_smape(y_pred, y_true)
            mql   = calculate_mql_sum(y_lower, y_pred, y_upper, y_true,
                                      lower_q=LOWER_Q, upper_q=UPPER_Q)
            mpir  = calculate_mpir(y_lower, y_upper)
            picp_val, _ = calculate_picp(y_lower, y_upper, y_true)
            cfe   = abs(COVERAGE - picp_val)

            all_rows.append({
                'model': model_name, 'prosumer': prosumer_name, 'target': label,
                'MAE': mae, 'R2': r2, 'sMAPE': smape,
                'MQL_sum': mql, 'MPIR': mpir, 'PICP': picp_val, 'CFE': cfe,
            })

        print(f'  {prosumer_name} done')

    return pd.DataFrame(all_rows)


In [ ]:
# ── Run: Power Production ──────────────────────────────────────────────────
print('Conference models — Production')
summary_prod = run_conference_scenario('Produced', 'Production')
summary_prod


In [ ]:
# ── Run: Power Consumption ─────────────────────────────────────────────────
print('Conference models — Consumption')
summary_cons = run_conference_scenario('Total Consumed', 'Consumption')
summary_cons


In [ ]:
# ── Table 3: average across prosumers per model × target ──────────────────
conf_summary = pd.concat([summary_prod, summary_cons], ignore_index=True)
METRIC_COLS = ['MAE', 'R2', 'sMAPE', 'MQL_sum', 'MPIR', 'CFE']
conf_summary_avg = (
    conf_summary
    .groupby(['model', 'target'])[METRIC_COLS]
    .mean()
    .round(4)
)
conf_summary.to_csv(os.path.join(CONF_RESULTS_DIR, 'conference_summary.csv'), index=False)
conf_summary_avg.to_csv(os.path.join(CONF_RESULTS_DIR, 'table3_reproduction.csv'))

print('Table 3 — Average metrics per model × target:')
with pd.option_context('display.float_format', '{:.4f}'.format):
    display(conf_summary_avg)


## 5c. §5.1 — Training-Set Size Sweep (Figure 3)

Evaluates all 5 single-step models on a **single Uppsala prosumer** (production)
as training-set size grows from 50 to 9 000 samples.

Metric: **MQL_Σ** (mean pinball loss, lower/median/upper at q=0.10/0.50/0.90).


In [ ]:
# ── §5.1 Training-size experiment ─────────────────────────────────────────
# Use the first Uppsala production prosumer (index 0 in conf_dfs for Uppsala)
# Filter to only Uppsala prosumers whose name starts with 'Uppsala'
uppsala_prod_dfs = [
    (name, df) for name, df in conf_dfs
    if 'Uppsala' in name and 'Produced' in df.columns
]

if not uppsala_prod_dfs:
    print('[skip] No Uppsala production dataframes found in conf_dfs')
else:
    # Use first prosumer for the sweep
    sweep_name, sweep_df_raw = uppsala_prod_dfs[0]
    print(f'Training-size sweep on: {sweep_name}  ({len(sweep_df_raw)} rows)')

    # Build model function map — wraps models_conference callables
    model_fn_map = {
        'LightGBM':      lgbm_singlestep_forecast,
        'CatBoost':      catboost_quantile_forecast,
        'CatBoost_Multi': catboost_multiquantile_forecast,
        'LinearQR':      linear_quantile_forecast,
    }
    if RUN_GP:
        model_fn_map['GP'] = gp_forecast

    sweep_results = training_size_sweep(
        sweep_df_raw, target_col='Produced',
        model_fn_map=model_fn_map,
        lower_q=LOWER_Q, upper_q=UPPER_Q,
    )
    sweep_results.to_csv(
        os.path.join(CONF_RESULTS_DIR, 'training_size_sweep.csv'), index=False
    )
    print(sweep_results.head(10))


In [ ]:
# ── Figure 3 plot ──────────────────────────────────────────────────────────
if 'sweep_results' in dir() and not sweep_results.empty:
    plot_training_size_sweep(
        sweep_results,
        title='MQL_Σ vs Training Set Size — Uppsala Production (Figure 3)',
        output_path=os.path.join(IMAGES_DIR, 'fig3_training_size_sweep.png'),
    )


## 5d. §5.2 — Model Aggregation & Generalisation (Figure 4)

Compares **Individual** (own prosumer data), **Grid** (all 7 Uppsala households combined),
and **LOO** (leave-one-out) training strategies for GBQR-LightGBM on Uppsala production.

Metric: MQL_Σ per household (boxplot across 7 prosumers).


In [ ]:
# ── §5.2 Aggregation scenarios ─────────────────────────────────────────────
if not uppsala_prod_dfs:
    print('[skip] No Uppsala production dataframes found')
else:
    agg_results = aggregation_scenarios(
        dfs=uppsala_prod_dfs,
        target_col='Produced',
        forecast_fn=lgbm_singlestep_forecast,
        lower_q=LOWER_Q,
        upper_q=UPPER_Q,
    )
    agg_results.to_csv(
        os.path.join(CONF_RESULTS_DIR, 'aggregation_scenarios.csv'), index=False
    )
    print(agg_results)
    plot_aggregation_scenarios(
        agg_results,
        title='MQL_Σ by Aggregation Scenario — Uppsala Production (Figure 4)',
        output_path=os.path.join(IMAGES_DIR, 'fig4_aggregation_scenarios.png'),
    )


## 5e. §5.3 — Auto-Correlation Analysis (Figure 5)

ACF at lags 1–24 for PV production for:
- **Uppsala** prosumer 1  
- **Halmstad** prosumer 4 (index 3)


In [ ]:
# ── §5.3 Auto-correlation ──────────────────────────────────────────────────
# Uppsala: prosumer 1  (index 0)
# Halmstad: prosumer 4  (index 3 in Halmstad-only list)
LAGS = 24

uppsala_prod  = [(name, df) for name, df in conf_dfs
                 if 'Uppsala' in name and 'Produced' in df.columns]
halmstad_prod = [(name, df) for name, df in conf_dfs
                 if 'Halmstad' in name and 'Produced' in df.columns]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Auto-Correlation — PV Production (Figure 5)', fontsize=14)

for ax, (site_name, site_dfs, prosumer_idx) in zip(
    axes,
    [('Uppsala', uppsala_prod, 0), ('Halmstad', halmstad_prod, 3)]
):
    if len(site_dfs) > prosumer_idx:
        selected = [site_dfs[prosumer_idx]]
        acf_mat  = autocorrelation_matrix(selected, 'Produced', lags=LAGS)
        im = plot_autocorrelation_heatmap(acf_mat, title=site_name, ax=ax)
        plt.colorbar(im, ax=ax, label='ACF')
    else:
        ax.set_title(f'{site_name} (no data)')

plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'fig5_autocorrelation.png'), dpi=150)
plt.show()


## 5f. §5.4 — Feature-Target Correlation (Figure 6)

Pearson, Spearman, and Kendall τ between weather features and each target
(Production and Consumption), averaged across all prosumers per site.

4-panel heatmap: Uppsala-Production · Uppsala-Consumption · Halmstad-Production · Halmstad-Consumption


In [ ]:
# ── §5.4 Feature-Target Correlation ──────────────────────────────────────
# Infer weather / feature columns from first DataFrame
sample_df = conf_dfs[0][1]
WEATHER_FEATURES = [c for c in sample_df.columns
                    if c not in ('Produced', 'Total Consumed', 'Date')]

PANEL_CONFIGS = [
    ('Uppsala',  'Produced',       'Uppsala — Production'),
    ('Uppsala',  'Total Consumed', 'Uppsala — Consumption'),
    ('Halmstad', 'Produced',       'Halmstad — Production'),
    ('Halmstad', 'Total Consumed', 'Halmstad — Consumption'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Feature-Target Correlation (Figure 6)', fontsize=14)

for ax, (site, target, title) in zip(axes.flat, PANEL_CONFIGS):
    site_dfs = [(n, d) for n, d in conf_dfs
                if site in n and target in d.columns]
    if not site_dfs:
        ax.set_title(f'{title} (no data)')
        continue
    corr = feature_target_correlation(site_dfs, target, WEATHER_FEATURES)
    # Show mean Pearson correlation as heatmap column
    vals = corr['pearson'][['mean']].values
    im = ax.imshow(vals, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks([0])
    ax.set_xticklabels(['Pearson'], fontsize=9)
    ax.set_yticks(range(len(WEATHER_FEATURES)))
    ax.set_yticklabels(WEATHER_FEATURES, fontsize=8)
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'fig6_feature_correlation.png'), dpi=150)
plt.show()


## 6. Evaluation Metrics (§5.6 Table 5)

Per-horizon metrics for all result CSVs in `Results/`:
**MAE, R², sMAPE (%), MQL_Σ, PICP, CFE, MPIR**

The `compute_all_metrics()` call below processes all multi-step strategy CSVs
(Direct / Recursive / Hybrid, Swedish + CityLearn).  For single-step Table 3,
see the summary produced in Section 5b above.


In [ ]:
compute_all_metrics(
    results_dir=RESULTS_DIR,
    output_dir=METRICS_DIR,
    forecast_steps=FORECAST_STEPS,
    coverage_target=COVERAGE,
)

In [ ]:
# Average metrics across all 48 horizon steps per forecasting configuration
summary = build_summary_table(METRICS_DIR)
summary.to_csv(os.path.join(METRICS_DIR, 'summary_table.csv'), index=False)

# Display
with pd.option_context('display.max_rows', 50, 'display.float_format', '{:.4f}'.format):
    display(summary)

## 7. Visualisation

### 7a. MAE / MPIR over forecast horizon (error plots)

In [ ]:
generate_all_error_plots(metrics_dir=METRICS_DIR, images_dir=IMAGES_DIR)

In [ ]:
# Display all Images/ plots in a grid
import glob
image_files = sorted(glob.glob(os.path.join(IMAGES_DIR, '*.png')))
image_files = [f for f in image_files if 'overview' not in f and 'sample' not in f]

n = len(image_files)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 5))
for ax, fpath in zip(axes.flat, image_files):
    ax.imshow(plt.imread(fpath))
    ax.set_title(os.path.basename(fpath).replace('.png', ''), fontsize=11)
    ax.axis('off')
for ax in axes.flat[n:]:
    ax.set_visible(False)
fig.suptitle('Error Metrics — MAE (blue) & MPIR (orange) over 48-h Horizon',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'all_plots_overview.png'), dpi=120, bbox_inches='tight')
plt.show()

### 7b. Single-sample forecast

In [ ]:
import glob
result_files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'direct_forecast_*.csv')))
if result_files:
    df_r  = pd.read_csv(result_files[0])
    row   = df_r.iloc[0]
    steps = FORECAST_STEPS
    preds   = row[[f'prediction_{i+1}'   for i in range(steps)]].values
    lowers  = row[[f'lower_bound_{i+1}'  for i in range(steps)]].values
    uppers  = row[[f'upper_bound_{i+1}'  for i in range(steps)]].values
    actuals = row[[f'actual_value_{i+1}' for i in range(steps)]].values
    plot_single_forecast(
        preds, lowers, uppers, actuals,
        row_idx=0,
        output_path=os.path.join(IMAGES_DIR, 'sample_forecast.png'),
    )
else:
    print('No result CSVs found — run Section 5 first.')